# JADES Photometric–Spectroscopic Catalog Matching

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from astropy.io import fits

# Define the project directories
PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

# Define the official JADES catalog paths
PHOTOMETRY_PATH = (
    RAW_DATA_DIR
    / "hlsp_jades_jwst_nircam_goods-n_photometry_v1.0_catalog.fits"
)

SPECTROSCOPY_PATH = (
    RAW_DATA_DIR
    / "hlsp_jades_jwst_nirspec_goods-n_prism-line-fluxes_v1.1_catalog.fits"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Photometry catalog exists: {PHOTOMETRY_PATH.exists()}")
print(f"Spectroscopy catalog exists: {SPECTROSCOPY_PATH.exists()}")

Project root: D:\AST\04_JADES_Catalog_Matching
Photometry catalog exists: True
Spectroscopy catalog exists: True


## 1. Inspecting the FITS catalog structure

A FITS file can contain multiple Header/Data Units (HDUs).

Each HDU may store:

- Primary metadata
- A binary catalog table
- Photometric measurements
- Quality flags
- Photometric-redshift estimates

Before loading the complete catalogs into pandas, we first inspect their structure and identify the scientifically relevant extensions.

In [3]:
from IPython.display import display


def summarize_fits_hdus(catalog_path):
    """Summarize FITS extensions without loading full tables into memory."""

    records = []

    with fits.open(catalog_path, memmap=True) as hdul:
        for index, hdu in enumerate(hdul):
            records.append(
                {
                    "index": index,
                    "name": hdu.name,
                    "hdu_type": type(hdu).__name__,
                    "rows": hdu.header.get("NAXIS2", 0),
                    "columns": hdu.header.get("TFIELDS", 0),
                }
            )

    return pd.DataFrame(records)


print("NIRCam photometric catalog")
display(summarize_fits_hdus(PHOTOMETRY_PATH))

print("NIRSpec spectroscopic catalog")
display(summarize_fits_hdus(SPECTROSCOPY_PATH))

NIRCam photometric catalog


,index,name,hdu_type,rows,columns
0,0,PRIMARY,PrimaryHDU,0,0
1,1,FILTERS,BinTableHDU,30,12
2,2,FLAG,BinTableHDU,85709,96
3,3,SIZE,BinTableHDU,85709,50
4,4,CIRC,BinTableHDU,85709,570
5,5,CIRC_BSUB,BinTableHDU,85709,570
6,6,CIRC_CONV,BinTableHDU,85709,570
7,7,KRON,BinTableHDU,85709,246
8,8,KRON_CONV,BinTableHDU,85709,246
9,9,PHOTOZ,BinTableHDU,85709,10


NIRSpec spectroscopic catalog


,index,name,hdu_type,rows,columns
0,0,PRIMARY,PrimaryHDU,0,0
1,1,Joined,BinTableHDU,1561,93


In [4]:
with fits.open(PHOTOMETRY_PATH, memmap=True) as phot_hdul:
    flag_columns = phot_hdul["FLAG"].columns.names
    circ_conv_columns = phot_hdul["CIRC_CONV"].columns.names
    photoz_columns = phot_hdul["PHOTOZ"].columns.names

with fits.open(SPECTROSCOPY_PATH, memmap=True) as spec_hdul:
    spectroscopy_columns = spec_hdul["Joined"].columns.names

flag_core_columns = [
    name
    for name in ["ID", "RA", "DEC", "FLAG_ST", "FLAG_BS", "FLAG_BN"]
    if name in flag_columns
]

print("FLAG core columns:")
print(flag_core_columns)

print("\nPHOTOZ columns:")
print(photoz_columns)

print(f"\nNumber of CIRC_CONV columns: {len(circ_conv_columns)}")

print(f"\nNumber of spectroscopy columns: {len(spectroscopy_columns)}")
print("First 30 spectroscopy columns:")
print(spectroscopy_columns[:30])

FLAG core columns:
['ID', 'RA', 'DEC', 'FLAG_ST', 'FLAG_BS', 'FLAG_BN']

PHOTOZ columns:
['ID', 'EAZY_z_a', 'EAZY_chisq_min', 'EAZY_l68', 'EAZY_u68', 'EAZY_l95', 'EAZY_u95', 'EAZY_l99', 'EAZY_u99', 'EAZY_nfilt']

Number of CIRC_CONV columns: 570

Number of spectroscopy columns: 93
First 30 spectroscopy columns:
['NIRSpec_ID', 'TIER', 'PID', 'Field', 'NIRCam_ID', 'RA_TARG', 'Dec_TARG', 'RA_NIRCam', 'Dec_NIRCam', 'Priority', 'z_Spec', 'z_Spec_flag', 'x_offset', 'y_offset', 'assigned_Prism', 'assigned_G140M', 'assigned_G235M', 'assigned_G395M', 'assigned_G395H', 'nDither_Pr', 'nDither_Gr', 'nInt_Prism', 'nInt_G140M', 'nInt_G235M', 'nInt_G395M', 'nInt_G395H', 'tExp_Prism', 'tExp_G140M', 'tExp_G235M', 'tExp_G395M']


In [5]:
FILTERS = [
    "F090W",
    "F115W",
    "F150W",
    "F200W",
    "F277W",
    "F335M",
    "F356W",
    "F410M",
    "F444W",
]

photometry_schema = []

for filter_name in FILTERS:
    flux_column = f"{filter_name}_CIRC2"
    nmad_error_column = f"{filter_name}_CIRC2_e"
    pipeline_error_column = f"{filter_name}_CIRC2_ei"

    photometry_schema.append(
        {
            "filter": filter_name,
            "flux_column": flux_column,
            "flux_exists": flux_column in circ_conv_columns,
            "nmad_error_column": nmad_error_column,
            "nmad_error_exists": nmad_error_column in circ_conv_columns,
            "pipeline_error_column": pipeline_error_column,
            "pipeline_error_exists": pipeline_error_column in circ_conv_columns,
        }
    )

photometry_schema = pd.DataFrame(photometry_schema)
display(photometry_schema)

,filter,flux_column,flux_exists,nmad_error_column,nmad_error_exists,pipeline_error_column,pipeline_error_exists
0,F090W,F090W_CIRC2,True,F090W_CIRC2_e,True,F090W_CIRC2_ei,True
1,F115W,F115W_CIRC2,True,F115W_CIRC2_e,True,F115W_CIRC2_ei,True
2,F150W,F150W_CIRC2,True,F150W_CIRC2_e,True,F150W_CIRC2_ei,True
3,F200W,F200W_CIRC2,True,F200W_CIRC2_e,True,F200W_CIRC2_ei,True
4,F277W,F277W_CIRC2,True,F277W_CIRC2_e,True,F277W_CIRC2_ei,True
5,F335M,F335M_CIRC2,True,F335M_CIRC2_e,True,F335M_CIRC2_ei,True
6,F356W,F356W_CIRC2,True,F356W_CIRC2_e,True,F356W_CIRC2_ei,True
7,F410M,F410M_CIRC2,True,F410M_CIRC2_e,True,F410M_CIRC2_ei,True
8,F444W,F444W_CIRC2,True,F444W_CIRC2_e,True,F444W_CIRC2_ei,True


In [6]:
with fits.open(PHOTOMETRY_PATH, memmap=True) as phot_hdul:
    flag_table = phot_hdul["FLAG"].data
    photoz_table = phot_hdul["PHOTOZ"].data

    photometry_preview = pd.DataFrame(
        {
            "phot_id": np.asarray(
                flag_table["ID"][:5]
            ).astype(np.int64),
            "ra_deg": np.asarray(
                flag_table["RA"][:5]
            ).astype(np.float64),
            "dec_deg": np.asarray(
                flag_table["DEC"][:5]
            ).astype(np.float64),
            "photoz_id": np.asarray(
                photoz_table["ID"][:5]
            ).astype(np.int64),
            "z_phot": np.asarray(
                photoz_table["EAZY_z_a"][:5]
            ).astype(np.float64),
        }
    )

display(photometry_preview)

ids_are_aligned = np.array_equal(
    photometry_preview["phot_id"],
    photometry_preview["photoz_id"],
)

print(f"FLAG and PHOTOZ IDs aligned: {ids_are_aligned}")

,phot_id,ra_deg,dec_deg,photoz_id,z_phot
0,1000003,189.130017,62.211833,1000003,4.34
1,1000009,189.130970,62.212145,1000009,1.47
2,1000011,189.125325,62.212237,1000011,1.51
3,1000012,189.131064,62.212308,1000012,1.07
4,1000015,189.127833,62.212449,1000015,1.52


FLAG and PHOTOZ IDs aligned: True


In [7]:
with fits.open(SPECTROSCOPY_PATH, memmap=True) as spec_hdul:
    spec_table = spec_hdul["Joined"].data

    spectroscopy_preview = pd.DataFrame(
        {
            "nirspec_id": np.asarray(
                spec_table["NIRSpec_ID"][:5]
            ).astype(np.int64),
            "nircam_id": np.asarray(
                spec_table["NIRCam_ID"][:5]
            ).astype(np.int64),
            "ra_nircam_deg": np.asarray(
                spec_table["RA_NIRCam"][:5]
            ).astype(np.float64),
            "dec_nircam_deg": np.asarray(
                spec_table["Dec_NIRCam"][:5]
            ).astype(np.float64),
            "z_spec": np.asarray(
                spec_table["z_Spec"][:5]
            ).astype(np.float64),
            "z_spec_quality": np.char.strip(
                np.asarray(spec_table["z_Spec_flag"][:5]).astype(str)
            ),
            "dr_problem": np.asarray(
                spec_table["DR_flag"][:5]
            ).astype(bool),
        }
    )

display(spectroscopy_preview)

,nirspec_id,nircam_id,ra_nircam_deg,dec_nircam_deg,z_spec,z_spec_quality,dr_problem
0,4,-9999,NaN,NaN,-1.000000,E,False
1,33,1000033,189.137063,62.213274,3.909444,A,False
2,58,1000058,189.131020,62.213989,2.442181,A,False
3,95,1000095,189.129203,62.215139,3.909191,A,False
4,110,1000110,189.146379,62.215508,4.064162,A,False


### Spectroscopic-redshift quality

The official JADES NIRSpec quality definitions are:

- **A**: highly robust redshift supported by high-S/N emission lines in medium-resolution spectra
- **B**: highly robust redshift supported by high-S/N emission lines in prism spectra
- **C**: secure redshift identified from spectral breaks and/or lower-S/N emission lines
- **D**: tentative redshift
- **E**: no reliable redshift

The main machine-learning sample will use quality classes A, B, and C, while retaining D and E in the complete audit table.

For `NIRCam_ID`:

- Positive values indicate an associated NIRCam source
- `-9999` indicates no NIRCam source within 0.2 arcsec
- `-1111` indicates a target outside the NIRCam footprint

with fits.open(SPECTROSCOPY_PATH, memmap=True) as spec_hdul:
    spec_table = spec_hdul["Joined"].data

    nircam_ids = np.asarray(
        spec_table["NIRCam_ID"]
    ).astype(np.int64)

    redshift_quality = np.char.strip(
        np.asarray(spec_table["z_Spec_flag"]).astype(str)
    )

    reduction_problem = np.asarray(
        spec_table["DR_flag"]
    ).astype(bool)

quality_counts = (
    pd.Series(redshift_quality, name="quality")
    .value_counts()
    .sort_index()
    .rename_axis("z_spec_quality")
    .reset_index(name="number_of_rows")
)

unmatched_id_counts = (
    pd.Series(nircam_ids[nircam_ids <= 0], name="nircam_id")
    .value_counts()
    .sort_index()
    .rename_axis("catalog_value")
    .reset_index(name="number_of_rows")
)

print(f"Total spectroscopy rows: {len(nircam_ids)}")
print(f"Rows with positive NIRCam ID: {(nircam_ids > 0).sum()}")
print(f"Rows with reduction problems: {reduction_problem.sum()}")

print("\nRedshift-quality counts:")
display(quality_counts)

print("Non-positive NIRCam ID counts:")
display(unmatched_id_counts)

In [8]:
with fits.open(SPECTROSCOPY_PATH, memmap=True) as spec_hdul:
    spec_table = spec_hdul["Joined"].data

    nircam_ids = np.asarray(
        spec_table["NIRCam_ID"]
    ).astype(np.int64)

    redshift_quality = np.char.strip(
        np.asarray(spec_table["z_Spec_flag"]).astype(str)
    )

    reduction_problem = np.asarray(
        spec_table["DR_flag"]
    ).astype(bool)

quality_counts = (
    pd.Series(redshift_quality, name="quality")
    .value_counts()
    .sort_index()
    .rename_axis("z_spec_quality")
    .reset_index(name="number_of_rows")
)

unmatched_id_counts = (
    pd.Series(nircam_ids[nircam_ids <= 0], name="nircam_id")
    .value_counts()
    .sort_index()
    .rename_axis("catalog_value")
    .reset_index(name="number_of_rows")
)

print(f"Total spectroscopy rows: {len(nircam_ids)}")
print(f"Rows with positive NIRCam ID: {(nircam_ids > 0).sum()}")
print(f"Rows with reduction problems: {reduction_problem.sum()}")

print("\nRedshift-quality counts:")
display(quality_counts)

print("Non-positive NIRCam ID counts:")
display(unmatched_id_counts)

Total spectroscopy rows: 1561
Rows with positive NIRCam ID: 1456
Rows with reduction problems: 66

Redshift-quality counts:


,z_spec_quality,number_of_rows
0,A,868
1,B,67
2,C,95
3,D,134
4,E,397


Non-positive NIRCam ID counts:


,catalog_value,number_of_rows
0,-9999,62
1,-1111,43


In [9]:
def numeric_column(table, column_name, dtype=np.float64):
    """Return a native-endian numeric copy of a FITS table column."""

    values = np.asarray(table[column_name])
    return values.astype(dtype, copy=True)


def text_column(table, column_name):
    """Return a stripped text copy of a FITS table column."""

    values = np.asarray(table[column_name])
    return np.char.strip(values.astype(str))

### Join

In [10]:
with fits.open(PHOTOMETRY_PATH, memmap=True) as phot_hdul:
    flag_table = phot_hdul["FLAG"].data
    photometry_table = phot_hdul["CIRC_CONV"].data
    photoz_table = phot_hdul["PHOTOZ"].data

    flag_ids = numeric_column(
        flag_table, "ID", dtype=np.int64
    )
    photometry_ids = numeric_column(
        photometry_table, "ID", dtype=np.int64
    )
    photoz_ids = numeric_column(
        photoz_table, "ID", dtype=np.int64
    )

    if not (
        np.array_equal(flag_ids, photometry_ids)
        and np.array_equal(flag_ids, photoz_ids)
    ):
        raise ValueError(
            "Photometric catalog extensions are not aligned by ID."
        )

    photometry_f200w = pd.DataFrame(
        {
            "phot_id": flag_ids,
            "ra_deg": numeric_column(flag_table, "RA"),
            "dec_deg": numeric_column(flag_table, "DEC"),
            "z_phot": numeric_column(photoz_table, "EAZY_z_a"),
            "flux_f200w_njy": numeric_column(
                photometry_table, "F200W_CIRC2"
            ),
            "fluxerr_f200w_nmad_njy": numeric_column(
                photometry_table, "F200W_CIRC2_e"
            ),
            "fluxerr_f200w_pipeline_njy": numeric_column(
                photometry_table, "F200W_CIRC2_ei"
            ),
        }
    )

print(f"Rows: {len(photometry_f200w):,}")
print(f"Columns: {photometry_f200w.shape[1]}")

display(photometry_f200w.head())

Rows: 85,709
Columns: 7


,phot_id,ra_deg,dec_deg,z_phot,flux_f200w_njy,fluxerr_f200w_nmad_njy,fluxerr_f200w_pipeline_njy
0,1000003,189.130017,62.211833,4.34,19.030500,5.183924,4.631001
1,1000009,189.130970,62.212145,1.47,19.930063,5.175885,4.619014
2,1000011,189.125325,62.212237,1.51,59.047279,5.339732,4.704514
3,1000012,189.131064,62.212308,1.07,14.172136,5.092314,3.870963
4,1000015,189.127833,62.212449,1.52,5.724563,3.235430,3.235538


In [11]:
with fits.open(PHOTOMETRY_PATH, memmap=True) as phot_hdul:
    flag_table = phot_hdul["FLAG"].data
    photometry_table = phot_hdul["CIRC_CONV"].data
    photoz_table = phot_hdul["PHOTOZ"].data

    catalog_columns = {
        "phot_id": numeric_column(
            flag_table, "ID", dtype=np.int64
        ),
        "ra_deg": numeric_column(
            flag_table, "RA"
        ),
        "dec_deg": numeric_column(
            flag_table, "DEC"
        ),
        "z_phot": numeric_column(
            photoz_table, "EAZY_z_a"
        ),
        "z_phot_chisq": numeric_column(
            photoz_table, "EAZY_chisq_min"
        ),
        "z_phot_n_filters": numeric_column(
            photoz_table, "EAZY_nfilt"
        ),
    }

    for filter_name in FILTERS:
        filter_key = filter_name.lower()

        catalog_columns[f"flux_{filter_key}_njy"] = numeric_column(
            photometry_table,
            f"{filter_name}_CIRC2",
        )

        catalog_columns[
            f"fluxerr_{filter_key}_nmad_njy"
        ] = numeric_column(
            photometry_table,
            f"{filter_name}_CIRC2_e",
        )

        catalog_columns[
            f"fluxerr_{filter_key}_pipeline_njy"
        ] = numeric_column(
            photometry_table,
            f"{filter_name}_CIRC2_ei",
        )

photometry_catalog = pd.DataFrame(catalog_columns)

invalid_z_phot = photometry_catalog["z_phot"] < 0
photometry_catalog.loc[invalid_z_phot, "z_phot"] = np.nan

In [12]:
finite_coordinates = (
    np.isfinite(photometry_catalog["ra_deg"])
    & np.isfinite(photometry_catalog["dec_deg"])
)

catalog_summary = pd.Series(
    {
        "rows": len(photometry_catalog),
        "columns": photometry_catalog.shape[1],
        "unique_source_ids": photometry_catalog["phot_id"].nunique(),
        "duplicate_source_ids": photometry_catalog[
            "phot_id"
        ].duplicated().sum(),
        "sources_with_finite_coordinates": finite_coordinates.sum(),
        "sources_with_z_phot": photometry_catalog[
            "z_phot"
        ].notna().sum(),
    },
    name="value",
)

display(catalog_summary.to_frame())

preview_columns = [
    "phot_id",
    "ra_deg",
    "dec_deg",
    "z_phot",
    "flux_f200w_njy",
    "fluxerr_f200w_nmad_njy",
    "flux_f444w_njy",
    "fluxerr_f444w_nmad_njy",
]

display(photometry_catalog[preview_columns].head())

,value
rows,85709
columns,33
unique_source_ids,85709
duplicate_source_ids,0
sources_with_finite_coordinates,85709
sources_with_z_phot,84820


,phot_id,ra_deg,dec_deg,z_phot,flux_f200w_njy,fluxerr_f200w_nmad_njy,flux_f444w_njy,fluxerr_f444w_nmad_njy
0,1000003,189.130017,62.211833,4.34,19.030500,5.183924,8.907708,3.490342
1,1000009,189.130970,62.212145,1.47,19.930063,5.175885,22.014006,3.492771
2,1000011,189.125325,62.212237,1.51,59.047279,5.339732,60.832451,3.548895
3,1000012,189.131064,62.212308,1.07,14.172136,5.092314,16.239920,3.498380
4,1000015,189.127833,62.212449,1.52,5.724563,3.235430,5.510203,3.249444


## 2. Loading the NIRSpec spectroscopy catalog

The repeated FITS parsing, native-endian conversion, redshift-quality selection, and catalog-summary logic are kept in `src/spectroscopy.py`. This keeps the notebook focused on the scientific inputs and results while leaving the complete implementation locally inspectable and reusable.

The module retains the identifiers, sky coordinates, redshift measurements, quality flags, reduction flags, and exposure information required for catalog matching.

In [14]:
import sys

# Make the local src package importable from the notebook directory
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dr3.spectroscopy import (
    load_spectroscopy_catalog,
    summarize_spectroscopy_catalog,
)

spectroscopy_catalog = load_spectroscopy_catalog(SPECTROSCOPY_PATH)
spectroscopy_summary, quality_summary = summarize_spectroscopy_catalog(
    spectroscopy_catalog
)

display(spectroscopy_catalog.head())
display(spectroscopy_summary)
display(quality_summary)

,nirspec_id,tier,program_id,catalog_nircam_id,ra_target_deg,dec_target_deg,ra_nircam_deg,dec_nircam_deg,z_spec,z_spec_quality,dr_problem,prism_flux_problem,prism_exposure_s,is_secure_spec
0,4,goods-n-mediumjwst,1181,-9999,189.132464,62.211894,NaN,NaN,NaN,E,False,False,6214.9,False
1,33,goods-n-mediumjwst,1181,1000033,189.137065,62.213273,189.137063,62.213274,3.909444,A,False,False,6214.9,True
2,58,goods-n-mediumjwst,1181,1000058,189.131012,62.213989,189.131020,62.213989,2.442181,A,False,False,9322.3,True
3,95,goods-n-mediumjwst,1181,1000095,189.129198,62.215141,189.129203,62.215139,3.909191,A,False,False,9322.3,True
4,110,goods-n-mediumjwst,1181,1000110,189.146379,62.215508,189.146379,62.215508,4.064162,A,False,False,9322.3,True


,value
spectroscopy_rows,1561
rows_with_positive_nircam_id,1456
unique_positive_nircam_ids,1453
duplicate_positive_id_rows,3
rows_with_finite_nircam_coordinates,1455
rows_without_nearby_nircam_source,62
rows_outside_nircam_footprint,43
rows_with_valid_z_spec,1167
secure_abc_rows_without_dr_problem,994


,rows,valid_redshifts,reduction_problems,unique_positive_nircam_ids
z_spec_quality,,,,
A,868,868,31,827
B,67,67,1,64
C,95,95,4,90
D,134,133,3,125
E,397,4,27,348


In [15]:
positive_id_spectra = spectroscopy_catalog.loc[
    spectroscopy_catalog["catalog_nircam_id"] > 0
].copy()

duplicate_id_mask = positive_id_spectra.duplicated(
    subset="catalog_nircam_id",
    keep=False,
)

duplicate_spectra = (
    positive_id_spectra.loc[duplicate_id_mask]
    .sort_values(
        ["catalog_nircam_id", "z_spec_quality"]
    )
)

duplicate_columns = [
    "catalog_nircam_id",
    "nirspec_id",
    "tier",
    "z_spec",
    "z_spec_quality",
    "dr_problem",
    "prism_exposure_s",
]

print(
    "Number of duplicated NIRCam sources:",
    duplicate_spectra["catalog_nircam_id"].nunique(),
)

print(
    "Number of spectroscopy rows involved:",
    len(duplicate_spectra),
)

display(duplicate_spectra[duplicate_columns])

Number of duplicated NIRCam sources: 3
Number of spectroscopy rows involved: 6


,catalog_nircam_id,nirspec_id,tier,z_spec,z_spec_quality,dr_problem,prism_exposure_s
229,1005591,3991,goods-n-mediumhst,10.605965,A,False,24859.5
271,1005591,5591,goods-n-mediumjwst,10.604423,A,False,9322.3
954,1030668,30667,goods-n-mediumjwst,NaN,E,False,3107.4
955,1030668,30668,goods-n-mediumjwst,NaN,E,False,3107.4
1491,1081942,81942,goods-n-mediumjwst,2.998952,A,False,6214.9
1555,1081942,10083470,goods-n-mediumjwst,NaN,E,False,3107.4


In [16]:
QUALITY_RANK = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4,
}

ranked_spectra = positive_id_spectra.copy()

ranked_spectra["quality_rank"] = (
    ranked_spectra["z_spec_quality"]
    .map(QUALITY_RANK)
    .fillna(99)
    .astype(int)
)

ranked_spectra = ranked_spectra.sort_values(
    [
        "catalog_nircam_id",
        "dr_problem",
        "quality_rank",
        "prism_exposure_s",
    ],
    ascending=[
        True,
        True,
        True,
        False,
    ],
    kind="stable",
)

best_spectrum_per_source = (
    ranked_spectra
    .drop_duplicates(
        subset="catalog_nircam_id",
        keep="first",
    )
    .drop(columns="quality_rank")
    .reset_index(drop=True)
)

removed_duplicate_rows = (
    len(positive_id_spectra)
    - len(best_spectrum_per_source)
)

print(f"Input positive-ID rows: {len(positive_id_spectra):,}")
print(f"Unique selected sources: {len(best_spectrum_per_source):,}")
print(f"Removed duplicate rows: {removed_duplicate_rows}")
print(
    "Selected NIRCam IDs are unique:",
    best_spectrum_per_source["catalog_nircam_id"].is_unique,
)

Input positive-ID rows: 1,456
Unique selected sources: 1,453
Removed duplicate rows: 3
Selected NIRCam IDs are unique: True


In [17]:
id_matched_catalog = photometry_catalog.merge(
    best_spectrum_per_source,
    left_on="phot_id",
    right_on="catalog_nircam_id",
    how="inner",
    validate="one_to_one",
)

id_match_summary = pd.Series(
    {
        "photometric_sources": len(photometry_catalog),
        "unique_spectroscopic_sources": len(
            best_spectrum_per_source
        ),
        "id_matched_sources": len(id_matched_catalog),
        "spectroscopic_ids_not_found": (
            len(best_spectrum_per_source)
            - len(id_matched_catalog)
        ),
    },
    name="value",
)

display(id_match_summary.to_frame())

display(
    id_matched_catalog[
        [
            "phot_id",
            "catalog_nircam_id",
            "ra_deg",
            "dec_deg",
            "ra_nircam_deg",
            "dec_nircam_deg",
            "z_phot",
            "z_spec",
            "z_spec_quality",
        ]
    ].head()
)

,value
photometric_sources,85709
unique_spectroscopic_sources,1453
id_matched_sources,1451
spectroscopic_ids_not_found,2


,phot_id,catalog_nircam_id,ra_deg,dec_deg,ra_nircam_deg,dec_nircam_deg,z_phot,z_spec,z_spec_quality
0,1000033,1000033,189.137063,62.213274,189.137063,62.213274,4.22,3.909444,A
1,1000058,1000058,189.131020,62.213989,189.131020,62.213989,2.38,2.442181,A
2,1000095,1000095,189.129203,62.215139,189.129203,62.215139,4.15,3.909191,A
3,1000110,1000110,189.146379,62.215508,189.146379,62.215508,4.12,4.064162,A
4,1000113,1000113,189.123835,62.215495,189.123835,62.215495,0.78,5.788500,A


In [18]:
from astropy.coordinates import SkyCoord
import astropy.units as u


MATCH_RADIUS_ARCSEC = 0.2

finite_match_coordinates = (
    np.isfinite(id_matched_catalog["ra_deg"])
    & np.isfinite(id_matched_catalog["dec_deg"])
    & np.isfinite(id_matched_catalog["ra_nircam_deg"])
    & np.isfinite(id_matched_catalog["dec_nircam_deg"])
)

id_matched_catalog["separation_arcsec"] = np.nan

photometric_coordinates = SkyCoord(
    ra=id_matched_catalog.loc[
        finite_match_coordinates, "ra_deg"
    ].to_numpy()
    * u.deg,
    dec=id_matched_catalog.loc[
        finite_match_coordinates, "dec_deg"
    ].to_numpy()
    * u.deg,
)

spectroscopic_coordinates = SkyCoord(
    ra=id_matched_catalog.loc[
        finite_match_coordinates, "ra_nircam_deg"
    ].to_numpy()
    * u.deg,
    dec=id_matched_catalog.loc[
        finite_match_coordinates, "dec_nircam_deg"
    ].to_numpy()
    * u.deg,
)

angular_separation = photometric_coordinates.separation(
    spectroscopic_coordinates
)

id_matched_catalog.loc[
    finite_match_coordinates, "separation_arcsec"
] = angular_separation.arcsec

within_match_radius = (
    id_matched_catalog["separation_arcsec"]
    <= MATCH_RADIUS_ARCSEC
)

coordinate_summary = pd.Series(
    {
        "id_matched_sources": len(id_matched_catalog),
        "sources_with_finite_coordinate_pairs": (
            finite_match_coordinates.sum()
        ),
        "sources_within_0.2_arcsec": within_match_radius.sum(),
        "sources_beyond_0.2_arcsec": (
            (
                id_matched_catalog["separation_arcsec"]
                > MATCH_RADIUS_ARCSEC
            ).sum()
        ),
    },
    name="value",
)

display(coordinate_summary.to_frame())

display(
    id_matched_catalog.loc[
        id_matched_catalog["separation_arcsec"]
        > MATCH_RADIUS_ARCSEC,
        [
            "phot_id",
            "catalog_nircam_id",
            "ra_deg",
            "dec_deg",
            "ra_nircam_deg",
            "dec_nircam_deg",
            "separation_arcsec",
            "z_spec",
            "z_spec_quality",
        ],
    ]
)

,value
id_matched_sources,1451
sources_with_finite_coordinate_pairs,1451
sources_within_0.2_arcsec,1450
sources_beyond_0.2_arcsec,1


,phot_id,catalog_nircam_id,ra_deg,dec_deg,ra_nircam_deg,dec_nircam_deg,separation_arcsec,z_spec,z_spec_quality
1173,1072560,1072560,189.290473,62.171308,189.29054,62.171239,0.269902,3.379845,A


### NN

In [19]:
valid_photometry = (
    photometry_catalog.loc[finite_coordinates]
    .reset_index(drop=True)
)

finite_spec_coordinates = (
    np.isfinite(best_spectrum_per_source["ra_nircam_deg"])
    & np.isfinite(best_spectrum_per_source["dec_nircam_deg"])
)

all_photometric_coordinates = SkyCoord(
    ra=valid_photometry["ra_deg"].to_numpy() * u.deg,
    dec=valid_photometry["dec_deg"].to_numpy() * u.deg,
)

spectroscopic_coordinates_for_matching = SkyCoord(
    ra=best_spectrum_per_source.loc[
        finite_spec_coordinates, "ra_nircam_deg"
    ].to_numpy()
    * u.deg,
    dec=best_spectrum_per_source.loc[
        finite_spec_coordinates, "dec_nircam_deg"
    ].to_numpy()
    * u.deg,
)

nearest_phot_rows, nearest_separations, _ = (
    spectroscopic_coordinates_for_matching.match_to_catalog_sky(
        all_photometric_coordinates
    )
)

nearest_sky_matches = (
    best_spectrum_per_source.loc[
        finite_spec_coordinates
    ]
    .copy()
    .reset_index(drop=True)
)

nearest_sky_matches["nearest_phot_id"] = (
    valid_photometry.iloc[
        nearest_phot_rows
    ]["phot_id"].to_numpy()
)

nearest_sky_matches["nearest_separation_arcsec"] = (
    nearest_separations.arcsec
)

nearest_sky_matches["nearest_id_agrees"] = (
    nearest_sky_matches["catalog_nircam_id"]
    == nearest_sky_matches["nearest_phot_id"]
)

matching_radii = [0.05, 0.10, 0.20, 0.30, 0.50]

radius_scan = pd.DataFrame(
    {
        "radius_arcsec": matching_radii,
        "nearest_matches": [
            (
                nearest_sky_matches["nearest_separation_arcsec"]
                <= radius
            ).sum()
            for radius in matching_radii
        ],
    }
)

radius_scan["fraction_of_finite_spec_coordinates"] = (
    radius_scan["nearest_matches"]
    / len(nearest_sky_matches)
)

display(radius_scan)

,radius_arcsec,nearest_matches,fraction_of_finite_spec_coordinates
0,0.05,1450,0.998623
1,0.10,1450,0.998623
2,0.20,1451,0.999311
3,0.30,1451,0.999311
4,0.50,1451,0.999311


In [20]:
available_photometric_ids = set(
    photometry_catalog["phot_id"].to_numpy()
)

nearest_sky_matches["published_id_in_photometry"] = (
    nearest_sky_matches["catalog_nircam_id"].isin(
        available_photometric_ids
    )
)

requires_review = (
    ~nearest_sky_matches["published_id_in_photometry"]
    | ~nearest_sky_matches["nearest_id_agrees"]
    | (
        nearest_sky_matches["nearest_separation_arcsec"]
        > MATCH_RADIUS_ARCSEC
    )
)

review_columns = [
    "catalog_nircam_id",
    "nearest_phot_id",
    "published_id_in_photometry",
    "nearest_id_agrees",
    "nearest_separation_arcsec",
    "z_spec",
    "z_spec_quality",
    "dr_problem",
]

review_candidates = (
    nearest_sky_matches.loc[
        requires_review,
        review_columns,
    ]
    .sort_values(
        "nearest_separation_arcsec",
        ascending=False,
    )
)

missing_coordinate_records = (
    best_spectrum_per_source.loc[
        ~finite_spec_coordinates,
        [
            "catalog_nircam_id",
            "nirspec_id",
            "z_spec",
            "z_spec_quality",
            "dr_problem",
        ],
    ]
)

print(
    "Records requiring ID/coordinate review:",
    len(review_candidates),
)

display(review_candidates)

print(
    "Positive-ID records without finite NIRCam coordinates:",
    len(missing_coordinate_records),
)

display(missing_coordinate_records)

Records requiring ID/coordinate review: 2


,catalog_nircam_id,nearest_phot_id,published_id_in_photometry,nearest_id_agrees,nearest_separation_arcsec,z_spec,z_spec_quality,dr_problem
1042,1054396,1054378,False,False,0.692914,NaN,E,False
1174,1072560,1183967,True,False,0.163638,3.379845,A,False


Positive-ID records without finite NIRCam coordinates: 1


,catalog_nircam_id,nirspec_id,z_spec,z_spec_quality,dr_problem
1452,10180348,38649,NaN,E,False


In [21]:
exact_separation_by_id = (
    id_matched_catalog
    .set_index("catalog_nircam_id")["separation_arcsec"]
)

candidate_matches = nearest_sky_matches.copy()

candidate_matches["exact_id_separation_arcsec"] = (
    candidate_matches["catalog_nircam_id"].map(
        exact_separation_by_id
    )
)

candidate_matches["accept_exact_id"] = (
    candidate_matches["published_id_in_photometry"]
    & candidate_matches[
        "exact_id_separation_arcsec"
    ].le(MATCH_RADIUS_ARCSEC)
)

candidate_matches["accept_sky_fallback"] = (
    ~candidate_matches["accept_exact_id"]
    & candidate_matches[
        "nearest_separation_arcsec"
    ].le(MATCH_RADIUS_ARCSEC)
)

candidate_matches["selected_phot_id"] = pd.Series(
    pd.NA,
    index=candidate_matches.index,
    dtype="Int64",
)

candidate_matches["selected_separation_arcsec"] = np.nan
candidate_matches["match_method"] = "unmatched"

exact_mask = candidate_matches["accept_exact_id"]

candidate_matches.loc[
    exact_mask, "selected_phot_id"
] = candidate_matches.loc[
    exact_mask, "catalog_nircam_id"
].astype("Int64")

candidate_matches.loc[
    exact_mask, "selected_separation_arcsec"
] = candidate_matches.loc[
    exact_mask, "exact_id_separation_arcsec"
]

candidate_matches.loc[
    exact_mask, "match_method"
] = "exact_id"

fallback_mask = candidate_matches["accept_sky_fallback"]

candidate_matches.loc[
    fallback_mask, "selected_phot_id"
] = candidate_matches.loc[
    fallback_mask, "nearest_phot_id"
].astype("Int64")

candidate_matches.loc[
    fallback_mask, "selected_separation_arcsec"
] = candidate_matches.loc[
    fallback_mask, "nearest_separation_arcsec"
]

candidate_matches.loc[
    fallback_mask, "match_method"
] = "nearest_sky"

accepted_matches = candidate_matches.loc[
    candidate_matches["selected_phot_id"].notna()
].copy()

accepted_matches["selected_phot_id"] = (
    accepted_matches["selected_phot_id"].astype(np.int64)
)

In [22]:
if not accepted_matches["selected_phot_id"].is_unique:
    raise ValueError(
        "Multiple spectra selected the same photometric source."
    )

final_matched_catalog = photometry_catalog.merge(
    accepted_matches,
    left_on="phot_id",
    right_on="selected_phot_id",
    how="inner",
    validate="one_to_one",
)

final_matched_catalog["separation_arcsec"] = (
    final_matched_catalog["selected_separation_arcsec"]
)

final_match_summary = pd.Series(
    {
        "unique_positive_spectroscopic_sources": len(
            best_spectrum_per_source
        ),
        "accepted_matches": len(final_matched_catalog),
        "exact_id_matches": (
            final_matched_catalog["match_method"]
            == "exact_id"
        ).sum(),
        "nearest_sky_fallback_matches": (
            final_matched_catalog["match_method"]
            == "nearest_sky"
        ).sum(),
        "unmatched_positive_spectroscopic_sources": (
            len(best_spectrum_per_source)
            - len(final_matched_catalog)
        ),
        "maximum_accepted_separation_arcsec": (
            final_matched_catalog[
                "separation_arcsec"
            ].max()
        ),
    },
    name="value",
)

display(final_match_summary.to_frame())

display(
    final_matched_catalog[
        [
            "phot_id",
            "catalog_nircam_id",
            "match_method",
            "separation_arcsec",
            "z_phot",
            "z_spec",
            "z_spec_quality",
            "is_secure_spec",
        ]
    ].head()
)

display(
    final_matched_catalog.loc[
        final_matched_catalog["match_method"]
        == "nearest_sky",
        [
            "phot_id",
            "catalog_nircam_id",
            "nearest_phot_id",
            "separation_arcsec",
            "z_spec",
            "z_spec_quality",
        ],
    ]
)

,value
unique_positive_spectroscopic_sources,1453.000000
accepted_matches,1451.000000
exact_id_matches,1450.000000
nearest_sky_fallback_matches,1.000000
unmatched_positive_spectroscopic_sources,2.000000
maximum_accepted_separation_arcsec,0.163638


,phot_id,catalog_nircam_id,match_method,separation_arcsec,z_phot,z_spec,z_spec_quality,is_secure_spec
0,1000033,1000033,exact_id,0.0,4.22,3.909444,A,True
1,1000058,1000058,exact_id,0.0,2.38,2.442181,A,True
2,1000095,1000095,exact_id,0.0,4.15,3.909191,A,True
3,1000110,1000110,exact_id,0.0,4.12,4.064162,A,True
4,1000113,1000113,exact_id,0.0,0.78,5.788500,A,True


,phot_id,catalog_nircam_id,nearest_phot_id,separation_arcsec,z_spec,z_spec_quality
1450,1183967,1072560,1183967,0.163638,3.379845,A


In [23]:
with fits.open(PHOTOMETRY_PATH, memmap=True) as phot_hdul:
    flag_table = phot_hdul["FLAG"].data

    flag_quality_catalog = pd.DataFrame(
        {
            "phot_id": numeric_column(
                flag_table, "ID", dtype=np.int64
            ),
            "flag_star_raw": numeric_column(
                flag_table, "FLAG_ST", dtype=np.uint16
            ),
            "flag_bright_star": numeric_column(
                flag_table, "FLAG_BS", dtype=np.int8
            ),
            "flag_bad_neighbor": numeric_column(
                flag_table, "FLAG_BN", dtype=np.int8
            ),
        }
    )

flag_quality_catalog["is_flagged_star"] = (
    flag_quality_catalog["flag_star_raw"] == 1
)

flag_value_summary = (
    flag_quality_catalog[
        [
            "flag_star_raw",
            "flag_bright_star",
            "flag_bad_neighbor",
        ]
    ]
    .melt(
        var_name="flag",
        value_name="value",
    )
    .groupby(
        ["flag", "value"]
    )
    .size()
    .reset_index(name="number_of_sources")
)

display(flag_value_summary)
display(flag_quality_catalog.head())

,flag,value,number_of_sources
0,flag_bad_neighbor,0,60208
1,flag_bad_neighbor,1,14071
2,flag_bad_neighbor,2,11430
3,flag_bright_star,0,83913
4,flag_bright_star,1,1796
5,flag_star_raw,1,275
6,flag_star_raw,32768,85434


,phot_id,flag_star_raw,flag_bright_star,flag_bad_neighbor,is_flagged_star
0,1000003,32768,0,0,False
1,1000009,32768,0,2,False
2,1000011,32768,0,0,False
3,1000012,32768,0,2,False
4,1000015,32768,0,0,False
